#  Notebook 3 — Model Evaluation (Faster R-CNN)
**Steps:** Load model → mAP evaluation → Per-class metrics → Confusion matrix → Loss curves

In [ ]:
# ============================================================
#  CONFIG
# ============================================================
import os

DATASET_BASE    = "dataset"
MODEL_LOAD_PATH = "saved_model/fasterrcnn_egypt.pth"  
CHECKPOINT_DIR  = "checkpoints"                       
IMG_SIZE        = 800    
BATCH_SIZE      = 4       
CONF_THRESH     = 0.5     # confidence threshold
IOU_THRESH      = 0.5     # IoU for mAP
NUM_WORKERS     = 0       #
EVAL_SUBSET     = 500     # images to evaluate on (None = all test)

CLASSES = [
    "traffic light", "traffic sign", "car", "person", "bus",
    "truck", "rider", "bike", "motor", "train", "banner", "tuktuk"
]
NUM_CLASSES = len(CLASSES) + 1

SPLITS = {
    "train": os.path.join(DATASET_BASE, "train", "train"),
    "val"  : os.path.join(DATASET_BASE, "val",   "val"),
    "test" : os.path.join(DATASET_BASE, "test",  "test"),
}
print(" Config ready!")

## Cell 1 — Load Model

In [ ]:
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

if os.path.exists(MODEL_LOAD_PATH):
    model.load_state_dict(torch.load(MODEL_LOAD_PATH, map_location=DEVICE))
    print(f"Loaded: {MODEL_LOAD_PATH}")
else:
    print(f" Model not found: {MODEL_LOAD_PATH}")

model.to(DEVICE)
model.eval()
print(f"   Device: {DEVICE}")

## Cell 2 — Load Test Dataset

In [ ]:
import json, random
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as T

random.seed(42)

class EgyptDetectionDataset(Dataset):
    def __init__(self, split_path, img_size=800):
        self.img_dir  = os.path.join(split_path, "images")
        self.ann_file = os.path.join(split_path, "annotations.json")
        self.img_size = img_size
        with open(self.ann_file) as f:
            coco = json.load(f)
        self.images = coco["images"]
        self.img_id_to_anns = {}
        for ann in coco["annotations"]:
            self.img_id_to_anns.setdefault(ann["image_id"], []).append(ann)

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        image    = Image.open(os.path.join(self.img_dir, img_info["file_name"])).convert("RGB")
        orig_w, orig_h = image.size
        image    = image.resize((self.img_size, self.img_size))
        scale_x  = self.img_size / orig_w; scale_y = self.img_size / orig_h
        image    = T.ToTensor()(image)
        anns     = self.img_id_to_anns.get(img_info["id"], [])
        boxes, labels = [], []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            x1,y1 = x*scale_x, y*scale_y
            x2,y2 = (x+w)*scale_x, (y+h)*scale_y
            x1,y1,x2,y2 = [max(0,min(v,self.img_size)) for v in [x1,y1,x2,y2]]
            if x2>x1 and y2>y1:
                boxes.append([x1,y1,x2,y2])
                labels.append(ann["category_id"]+1)
        if boxes:
            boxes  = torch.as_tensor(boxes,  dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
        else:
            boxes  = torch.zeros((0,4), dtype=torch.float32)
            labels = torch.zeros((0,),  dtype=torch.int64)
        return image, {"boxes": boxes, "labels": labels, "image_id": torch.tensor([idx])}

TEST_DATASET = EgyptDetectionDataset(SPLITS["test"], img_size=IMG_SIZE)
if EVAL_SUBSET and EVAL_SUBSET < len(TEST_DATASET):
    TEST_DATASET = Subset(TEST_DATASET, random.sample(range(len(TEST_DATASET)), EVAL_SUBSET))

TEST_LOADER = DataLoader(TEST_DATASET, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=lambda b: tuple(zip(*b)),
                          num_workers=NUM_WORKERS, pin_memory=True)
print(f" Test set: {len(TEST_DATASET)} images")

## Cell 3 — Compute TP / FP / FN per Class

In [ ]:
import torch
import numpy as np
from tqdm import tqdm
from collections import defaultdict

def compute_iou(box1, box2):
    xA = max(box1[0], box2[0]); yA = max(box1[1], box2[1])
    xB = min(box1[2], box2[2]); yB = min(box1[3], box2[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    return inter / (a1 + a2 - inter + 1e-6)

tp = defaultdict(int)
fp = defaultdict(int)
fn = defaultdict(int)

for images, targets in tqdm(TEST_LOADER, desc="Evaluating"):
    images = [img.to(DEVICE) for img in images]
    with torch.no_grad():
        preds = model(images)

    for pred, target in zip(preds, targets):
        # Filter by confidence
        mask       = pred['scores'] >= CONF_THRESH
        pred_boxes = pred['boxes'][mask].cpu().numpy()
        pred_lbls  = pred['labels'][mask].cpu().numpy()
        gt_boxes   = target['boxes'].numpy()
        gt_lbls    = target['labels'].numpy()

        matched_gt = set()
        for pb, pl in zip(pred_boxes, pred_lbls):
            cls_id = int(pl) - 1  # back to 0-indexed
            best_iou, best_j = 0, -1
            for j, (gb, gl) in enumerate(zip(gt_boxes, gt_lbls)):
                if int(gl)-1 == cls_id and j not in matched_gt:
                    v = compute_iou(pb, gb)
                    if v > best_iou: best_iou, best_j = v, j
            if best_iou >= IOU_THRESH and best_j >= 0:
                tp[cls_id] += 1; matched_gt.add(best_j)
            else:
                fp[cls_id] += 1
        for j, gl in enumerate(gt_lbls):
            if j not in matched_gt:
                fn[int(gl)-1] += 1

print("\n Evaluation complete!")

## Cell 4 — Metrics Table

In [ ]:
import numpy as np

print(f"{'Class':<15} {'TP':>6} {'FP':>6} {'FN':>6} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("-" * 62)
precisions, recalls, f1s = [], [], []
for i, cls in enumerate(CLASSES):
    p  = tp[i]/(tp[i]+fp[i]+1e-6)
    r  = tp[i]/(tp[i]+fn[i]+1e-6)
    f1 = 2*p*r/(p+r+1e-6)
    precisions.append(p); recalls.append(r); f1s.append(f1)
    print(f"{cls:<15} {tp[i]:>6} {fp[i]:>6} {fn[i]:>6} {p:>10.3f} {r:>8.3f} {f1:>8.3f}")
print("-" * 62)
print(f"{'MEAN':<15} {'':>6} {'':>6} {'':>6} {np.mean(precisions):>10.3f} {np.mean(recalls):>8.3f} {np.mean(f1s):>8.3f}")

## Cell 5 — Per-Class Bar Chart

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.arange(len(CLASSES)); w = 0.25
fig, ax = plt.subplots(figsize=(15,5))
ax.bar(x-w,   precisions, w, label='Precision', color='steelblue', alpha=0.85)
ax.bar(x,     recalls,    w, label='Recall',    color='coral',     alpha=0.85)
ax.bar(x+w,   f1s,        w, label='F1',        color='green',     alpha=0.85)
ax.axhline(np.mean(precisions), color='steelblue', linestyle='--', linewidth=1, alpha=0.5)
ax.axhline(np.mean(recalls),    color='coral',     linestyle='--', linewidth=1, alpha=0.5)
ax.set_xticks(x); ax.set_xticklabels(CLASSES, rotation=35, ha='right', fontsize=10)
ax.set_ylim(0,1.1); ax.set_ylabel("Score")
ax.set_title(f"Per-Class Metrics — Faster R-CNN (conf≥{CONF_THRESH}, IoU≥{IOU_THRESH})")
ax.legend()
plt.tight_layout()
plt.savefig("per_class_metrics.png", dpi=150, bbox_inches='tight')
plt.show()
print(" Saved: per_class_metrics.png")

## Cell 6 — Confusion Matrix

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

y_true, y_pred = [], []
for i, cls in enumerate(CLASSES):
    y_true += [cls]*tp[i] + [cls]*fn[i] + ["None"]*fp[i]
    y_pred += [cls]*tp[i] + ["None"]*fn[i] + [cls]*fp[i]

cm_labels = CLASSES + ["None"]
cm = confusion_matrix(y_true, y_pred, labels=cm_labels)

fig, ax = plt.subplots(figsize=(15, 13))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=cm_labels, yticklabels=cm_labels,
            square=True, linewidths=0.5, annot_kws={'size':9}, ax=ax)
ax.set_title("Confusion Matrix — Faster R-CNN", fontsize=15)
ax.set_xlabel("Predicted", fontsize=12); ax.set_ylabel("Ground Truth", fontsize=12)
ax.set_xticklabels(cm_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(cm_labels, rotation=0,  fontsize=9)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()
print(" Saved: confusion_matrix.png")

## Cell 7 — Loss Curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

if os.path.exists("loss_curves.png"):
    img = mpimg.imread("loss_curves.png")
    plt.figure(figsize=(12,6))
    plt.imshow(img); plt.axis('off')
    plt.title("Training Loss Curves")
    plt.show()
else:
    print("loss_curves.png not found — run training first")